In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GINEConv
from torch_geometric.data import Batch
import os
from dataclasses import asdict, replace

try:
    from edge_feature_ablation.edge_features import (
        ABLATION_PRESETS,
        build_bidirectional_edges,
        edge_feature_names,
    )
    from edge_feature_ablation.experiment_config import MODEL_EXPERIMENTS
except ModuleNotFoundError:
    # When the kernel working directory is this notebook's directory.
    from edge_features import (
        ABLATION_PRESETS,
        build_bidirectional_edges,
        edge_feature_names,
    )
    from experiment_config import MODEL_EXPERIMENTS


In [ ]:
## 2
SEED = int(os.environ.get("TENNIS_SEED", "42"))
np.random.seed(SEED)
torch.manual_seed(SEED)

# relativ zu mac oder nvidia gpus
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
# device = torch.device("mps")
device


# Change only this environment variable between ablations.  All experimental
# constants below remain identical to the source experiment.
MODEL_EXPERIMENT = os.environ.get("TENNIS_MODEL_EXPERIMENT", "edge_full")
if MODEL_EXPERIMENT not in MODEL_EXPERIMENTS:
    raise ValueError(
        f"Unknown model experiment {MODEL_EXPERIMENT!r}; "
        f"choose one of {tuple(MODEL_EXPERIMENTS)}"
    )
MODEL_CONFIG = MODEL_EXPERIMENTS[MODEL_EXPERIMENT]

ABLATION_NAME = os.environ.get(
    "TENNIS_EDGE_ABLATION", MODEL_CONFIG.edge_preset
)
if ABLATION_NAME not in ABLATION_PRESETS:
    raise ValueError(
        f"Unknown ablation {ABLATION_NAME!r}; "
        f"choose one of {tuple(ABLATION_PRESETS)}"
    )
EDGE_CONFIG = ABLATION_PRESETS[ABLATION_NAME]
if MODEL_CONFIG.tournament_context_edges:
    EDGE_CONFIG = replace(EDGE_CONFIG, include_tournament_context=True)

# Manual switch for the tournament population used by both the GNN and the
# final logistic-regression baseline.
TOURNAMENT_SCOPE = os.environ.get(
    "TENNIS_TOURNAMENT_SCOPE", "slams_masters"
)  # "slams" or "slams_masters"
if TOURNAMENT_SCOPE not in {"slams", "slams_masters"}:
    raise ValueError(
        "TOURNAMENT_SCOPE must be 'slams' or 'slams_masters'"
    )

print("model experiment:", MODEL_EXPERIMENT)
print("model config:", asdict(MODEL_CONFIG))
print("edge ablation:", ABLATION_NAME)
print("edge features:", edge_feature_names(EDGE_CONFIG))
print("tournament scope:", TOURNAMENT_SCOPE)


In [ ]:
## 3
DATA_DIR = Path("../data")
PROCESSED_DIR = Path("../data/processed")
MATCHES_FILENAME = (
    "atp_matches_slams.csv"
    if TOURNAMENT_SCOPE == "slams"
    else "atp_matches_slams_masters.csv"
)
MATCHES_PATH = Path(PROCESSED_DIR / MATCHES_FILENAME)


In [ ]:
##4
## clean up data formats and load into a df, one row one match

matches = pd.read_csv(MATCHES_PATH)

matches["tourney_date"] = pd.to_datetime(matches["tourney_date"])



round_order_map = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}

matches["round_order"] = matches["round"].map(round_order_map)

matches = matches.sort_values(["tourney_date", "tourney_id", "round_order"]).reset_index(drop= True)


print(matches.shape)
matches.head

In [ ]:
INTRA_PATH = PROCESSED_DIR / "match_intransitivity_scores_with_levels.csv"

intra = pd.read_csv(INTRA_PATH)
intra["tourney_date"] = pd.to_datetime(intra["tourney_date"])

# Columns that identify the same match in both datasets.
merge_keys = [
    "tourney_id",
    "tourney_date",
    "round",
    "surface",
    "winner_name",
    "loser_name"
]

# We only merge the columns we actually need.
intra_cols = merge_keys + [
    "I_star_context",
    "I_star_direct",
    "intransitivity_level",
    "intransitivity_level_num"
]

matches = matches.merge(
    intra[intra_cols],
    on=merge_keys,
    how="left"
)

# If some Slam matches did not receive an intransitivity label,
# we mark them explicitly instead of leaving NaN.
matches["intransitivity_level"] = matches["intransitivity_level"].fillna("missing")
matches["intransitivity_level_num"] = matches["intransitivity_level_num"].fillna(-2).astype(int)

print(matches["intransitivity_level"].value_counts(dropna=False))

In [ ]:
##5
bscore_general = pd.read_csv(PROCESSED_DIR / "bscore_snapshots.csv")
bscore_hard = pd.read_csv(PROCESSED_DIR / "bscore_snapshots_hard.csv")
bscore_clay = pd.read_csv(PROCESSED_DIR / "bscore_snapshots_clay.csv")
bscore_grass = pd.read_csv(PROCESSED_DIR / "bscore_snapshots_grass.csv")

## csv's that act like dictionaries for us, having snapshots of the bscores for all players at every point in time, that is relevant


In [ ]:
##6
## here we merge all the "dictionaries of the csvs" into a single dataframe for future use

bscore_snapshots = bscore_general.copy()

if bscore_hard is not None:
    bscore_snapshots = bscore_snapshots.merge(
        bscore_hard,
        on=["tourney_id", "round_order", "player"],
        how = "left"
        )
    
else: 
    bscore_snapshots["bscore_hard"] = np.nan

if bscore_clay is not None:
    bscore_snapshots = bscore_snapshots.merge(
        bscore_clay,
        on=["tourney_id", "round_order", "player"],
        how = "left"
    )
else: 
    bscore_snapshots["bscore_clay"] = np.nan

if bscore_grass is not None:
    bscore_snapshots = bscore_snapshots.merge(
        bscore_grass,
        on=["tourney_id", "round_order", "player"],
        how = "left"
        )
else: 
    bscore_snapshots["bscore_grass"] = np.nan

bscore_snapshots.head()

In [ ]:
##7
for col in ["bscore_hard", "bscore_clay", "bscore_grass"]:
    bscore_snapshots[col] = bscore_snapshots[col].fillna(bscore_snapshots["bscore_general"])

bscore_snapshots.isna().sum()

In [ ]:
##8
def surface_one_hot(surface):
    if surface == "Hard":
        return[1.0, 0.0, 0.0]
    elif surface =="Clay":
        return[0.0, 1.0, 0.0]
    elif surface == "Grass":
        return[0.0, 0.0, 1.0]
    else:  return [0.0, 0.0, 0.0]

In [ ]:
## Score parsing and status handling live in edge_features.py.
# Keeping this cell makes the generated notebook structure align with the
# source experiment while ensuring every ablation uses the tested parser.


In [ ]:
##10
## a dataframe of all the players with no nans

## alle spieler die als gewinner im Dataset sind
winner_static_matches = matches[
    ["winner_name", "winner_ht", "winner_hand", "tourney_date"]
].rename(columns={"winner_name": "player",
                  "winner_ht": "height",
                  "winner_hand": "hand"})

## alle loser im Dataset
loser_static_matches = matches[
    ["loser_name", "loser_ht", "loser_hand", "tourney_date"]
].rename(columns={"loser_name": "player",
                  "loser_ht": "height",
                  "loser_hand": "hand"})

## beide df zusammen
player_static_df = pd.concat([winner_static_matches, loser_static_matches], ignore_index=True)
player_static_df = (
    player_static_df
    .sort_values("tourney_date")
    .groupby("player")
    .last()
)

## median der höhe der spieler um nan zu füllen
median_height = player_static_df["height"].dropna().median()
player_static_df["height"] = player_static_df["height"].fillna(median_height)

## eher rechts als linkshändler
player_static_df["hand"] = player_static_df["hand"].fillna("R")

player_static_df.head()

In [ ]:
## diese cell konstruiert die rolling blocks damit wir sie nicht im training loop machen müssen
## wir brauchen als Kontext auch die Matches pre Training Loop, ansonsten sind die embeddings viel zu 
# "schlecht"

START_DATE = "2006-01-01"

TRAIN_END = 2015
VAL_END = 2016

## future matches sind wo training beginnen wird
future_match = matches[matches["tourney_date"] >= "2011-01-01"].copy()
future_matches = future_match[future_match["tourney_date"] < "2021-01-01"].copy()

## tabelle mit rows tipo: Australian Open 2011 1
## drop duplicates weil viele same round matches in einem Turnier logisch sind
rolling_blocks = (
    future_matches[["tourney_date", "tourney_id", "round_order"]]
    .drop_duplicates()
    .sort_values(["tourney_date", "tourney_id", "round_order"])
    .reset_index(drop = True)
)

def assign_phase(year):
    if year <= 2010:
        return "warmup"
    elif year <= TRAIN_END:
        return "train"
    elif year <= VAL_END:
        return "val"
    else:
        return "test"

## redundant technically
# rolling_blocks = rolling_blocks[
#     rolling_blocks["tourney_date"].dt.year >= 2006
# ].reset_index(drop = True)

## 2 new columns
rolling_blocks["year"] = rolling_blocks["tourney_date"].dt.year
rolling_blocks["phase"] = rolling_blocks["tourney_date"].dt.year.apply(assign_phase)

print(rolling_blocks["phase"].value_counts())
rolling_blocks.head()

## Konstruktion rolling blocks
# jedes block eine runde
# im training arbeiten wir immer nur am current block
# graph is gebaut von training blocks und der history, und nichts was noch nicht geschehen ist

In [ ]:
SCORE_COLS = [
    "bscore_general",
    "bscore_hard",
    "bscore_clay",
    "bscore_grass"
]

bscore_index = {}
for (t_id, r_ord), group in bscore_snapshots.groupby(["tourney_id", "round_order"]):
    bscore_index[(t_id, r_ord)] = group.set_index("player")[SCORE_COLS].to_dict("index")


## die funktion gibt die bscore snapshots zurück indem man sie auf die Informationen des rolling_blocks aufruft
## es berechnet auch die defaults wenn vielleicht bei einem spieler die values mssing sind
def get_bscore_snapshots(tourney_id, round_order):
    snap_dict = bscore_index.get((tourney_id, round_order), {})

    ## passiert nicht aber good practice ig
    # missing_cols = [col for col in SCORE_COLS if col not in snap.columns]
    # if missing_cols:
    #     raise ValueError(f"Missing bscore columns: {missing_cols}")

    ## wenn der snapshot nicht existiert (ist nie der Fall) dann ist bscore 0.0 für alle, 
    # vielleicht median über ganzes bscore besser als 0.0
    # if snap.empty:
    #     defaults = {col: 0.0 for col in SCORE_COLS}
    #     return {}, defaults

    defaults = {}
    for col in SCORE_COLS:
        values = [
            float(v[col])
            for v in snap_dict.values()
            if col in v and not pd.isna(v[col])
        ]
        if values:
            defaults[col] = float(np.quantile(values, 0.25))
        else:
            defaults[col] = 0.0 

    # all_vals = {col: [v.get(col, 0.0) for v in snap_dict.values()] for col in SCORE_COLS}
    # defaults = {col: float(np.quantile(all_vals[col], 0.25)) if all_vals[col] else 0.0 for col in SCORE_COLS}
    return snap_dict, defaults
    # spieler und seine bscore werden zusammen im dictionary gespeichert

## für jedes block nehmen wir die bscore die wir bereits im vorherein berechnet haben und returnen nur diese und default

In [ ]:
## wir builden den GNN graph eines einzelnen blocks
## als input geben wir die history bis zu block, die matches des blocks, und alle bscore vor den matches
## las output bekommen wir den "data" obj von PyG und ein player_idx um ein mapping von name zu indizes zu haben
## kein leakage von den matches innerhalb des blocks weil graph auf die alten gebaut wird, dann prediction -> loop

def build_pyg_snapshot(current_history, block_matches, t_block, snap_dict, defaults, alpha_days=365):

    MAX_HISTORY_DAYS = 365 * 3  # usa solo ultimi 3 anni
    cutoff = t_block - pd.Timedelta(days=MAX_HISTORY_DAYS)
    current_history_trimmed = current_history[current_history["tourney_date"] >= cutoff]
    ## alle spieler die in dem block bis jetzt auftauchen
    all_players = pd.Index(
        pd.unique(
            pd.concat(
                [
                    current_history_trimmed["winner_name"],
                    current_history_trimmed["loser_name"],
                    block_matches["winner_name"],
                    block_matches["loser_name"],
                ],
                ignore_index=True
            )
        )
    )

    ## für alle spieler anstatt namen auch eine Zahl, damit man nicht mit strings arbeitet
    player_to_idx = {player: i for i, player in enumerate(all_players)}

    node_features = []

    ## dieses Loop nimmt die features von jedem Spieler aktualisiert auf Zeitpunkt
    ## und buildet dann die tensoren damit für die GNN
    for player in all_players:
        scores = snap_dict.get(player, defaults)

        bsg = float(scores.get("bscore_general", defaults["bscore_general"]))
        bsh = float(scores.get("bscore_hard", defaults["bscore_hard"]))
        bsc = float(scores.get("bscore_clay", defaults["bscore_clay"]))
        bsgra = float(scores.get("bscore_grass", defaults["bscore_grass"]))

        if player in player_static_df.index:
            height = player_static_df.loc[player, "height"]
            hand = player_static_df.loc[player, "hand"]
        else:
            height = median_height
            hand = "R"

        if pd.isna(height):
            height = median_height

        
        is_right = 1.0 if hand == "R" else 0.0

        node_features.append([
            bsg,
            bsh,
            bsc,
            bsgra,
            height,
            is_right
        ])

    x = torch.tensor(node_features, dtype=torch.float)
    raw_bscore_general = x[:, 0].clone()
    if not MODEL_CONFIG.node_bscore_features:
        # Keep the architecture fixed while removing general and
        # surface-specific B-score information from message passing.
        x[:, :4] = 0.0
    if MODEL_CONFIG.normalize_node_features and x.shape[0] > 1:
        # Continuous node features only; the hand flag remains binary.
        continuous = x[:, :5]
        feature_mean = continuous.mean(dim=0, keepdim=True)
        feature_std = continuous.std(dim=0, keepdim=True, unbiased=False)
        feature_std = torch.where(
            feature_std < 1e-6,
            torch.ones_like(feature_std),
            feature_std,
        )
        x[:, :5] = (continuous - feature_mean) / feature_std

    edge_pairs, edge_attributes, edge_stats = build_bidirectional_edges(
        matches=current_history_trimmed.to_dict("records"),
        player_to_idx=player_to_idx,
        snapshot_date=t_block,
        config=EDGE_CONFIG,
        alpha_days=alpha_days,
    )

    edge_dim = len(edge_feature_names(EDGE_CONFIG))
    if len(edge_pairs) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, edge_dim), dtype=torch.float)
    else:
        edge_index = torch.tensor(
            edge_pairs, dtype=torch.long
        ).t().contiguous()
        edge_attr = torch.tensor(edge_attributes, dtype=torch.float)

    ## NN's wissen nicht was zu tun mit nan's also im fall auf 0 setzen
    x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    edge_attr = torch.nan_to_num(edge_attr, nan=0.0, posinf=0.0, neginf=0.0)

    ## das eigentliche Graph was für den message passing gebraucht ist
    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        raw_bscore_general=torch.nan_to_num(
            raw_bscore_general, nan=0.0, posinf=0.0, neginf=0.0
        )
    )

    return data, player_to_idx

In [ ]:
def get_surface_bscore(scores, surface):
    if surface == "Hard":
        return float(scores.get("bscore_hard", scores.get("bscore_general", 0.0)))
    elif surface == "Clay":
        return float(scores.get("bscore_clay", scores.get("bscore_general", 0.0)))
    elif surface== "Grass":
        return float(scores.get("bscore_grass", scores.get("bscore_general", 0.0)))
    else:
        return float(scores.get("bscore_general", 0.0))
        
## diese funktion soll uns die targets randomizen, damit man nicht immer winner als player1 hat

## nimmt die current matches und wandelt sie in ein tensor um fürs modell benutzbar zu sein
## praktisch graph gibt es schon das sind die examples fürs Training
## paar lines sind von experimente die man nicht braucht
def make_block_targets(block_matches, player_to_idx, rng, snap_dict, defaults):

    rows = []

    for _, row in block_matches.iterrows():
        winner = row["winner_name"]
        loser= row["loser_name"]

        if winner not in player_to_idx or loser not in player_to_idx: 
            continue

        winner_idx = player_to_idx[winner]
        loser_idx = player_to_idx[loser]

        winner_scores = snap_dict.get(winner, defaults)
        loser_scores = snap_dict.get(loser, defaults)

        winner_bscore = float(winner_scores.get("bscore_general", defaults["bscore_general"]))
        loser_bscore = float(loser_scores.get("bscore_general", defaults["bscore_general"]))

        match_surface = surface_one_hot(row["surface"])

        round_ord = row["round_order"]
        ## scaling damit die daten zwischen 0 und 1 liegen
        round_scaled = float(round_ord) / 7.0 if pd.notna(round_ord) else 0.0


        if rng.random() < 0.5:
            player_a_idx = winner_idx
            player_b_idx = loser_idx
            y = 1.0
            bscore_diff = winner_bscore - loser_bscore
        else:
            player_a_idx = loser_idx
            player_b_idx = winner_idx
            y = 0.0
            bscore_diff = loser_bscore - winner_bscore

        match_context = match_surface + [round_scaled]
        

        rows.append({
            "player_a_idx": player_a_idx,
            "player_b_idx": player_b_idx,
            "match_context": match_context,
            "y": y,
            "bscore_diff": bscore_diff,
            "intransitivity_level": row["intransitivity_level"]
        })

    if len(rows) == 0:
        return None
    
    player_a = torch.tensor([r["player_a_idx"] for r in rows], dtype=torch.long)
    player_b = torch.tensor([r["player_b_idx"] for r in rows], dtype=torch.long)
    context = torch.tensor([r["match_context"] for r in rows], dtype=torch.float)
    y = torch.tensor([r["y"] for r in rows], dtype=torch.float)
    bscore_diffs = torch.tensor([r["bscore_diff"] for r in rows], dtype=torch.float)
    intransitivity_levels = [
        r["intransitivity_level"]
        for r in rows
    ]

    return player_a, player_b, context, y, bscore_diffs, intransitivity_levels

In [ ]:
class TennisGINE(nn.Module):
    def __init__(
        self,
        node_in_dim,
        edge_in_dim,
        hidden_dim,
        match_context_dim,
        dropout=0.5,
        init_scale=1.0,
    ):
        super().__init__()
        self.node_encoder = nn.Linear(node_in_dim, hidden_dim)

        def message_mlp():
            return nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )

        self.conv1 = GINEConv(
            message_mlp(),
            edge_dim=edge_in_dim,
            aggr=MODEL_CONFIG.aggregation,
        )
        self.conv2 = GINEConv(
            message_mlp(),
            edge_dim=edge_in_dim,
            aggr=MODEL_CONFIG.aggregation,
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        pair_dim = hidden_dim * 4
        self.match_head = nn.Sequential(
            nn.Linear(pair_dim + match_context_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.bscore_scale = nn.Parameter(torch.tensor(float(init_scale)))
        nn.init.zeros_(self.match_head[-1].weight)
        nn.init.zeros_(self.match_head[-1].bias)

    @staticmethod
    def pair_features(h_a, h_b):
        return torch.cat(
            [h_a, h_b, h_a - h_b, torch.abs(h_a - h_b)],
            dim=1,
        )

    def pair_score(self, h_a, h_b, match_context):
        pair_z = self.pair_features(h_a, h_b)
        return self.match_head(
            torch.cat([pair_z, match_context], dim=1)
        ).squeeze(-1)

    def forward(self, data, player_a_idx, player_b_idx, match_context):
        h0 = self.node_encoder(data.x)

        message1 = self.norm1(
            F.relu(self.conv1(h0, data.edge_index, data.edge_attr))
        )
        h1 = h0 + message1 if MODEL_CONFIG.residual_connections else message1

        message2 = self.norm2(
            F.relu(self.conv2(h1, data.edge_index, data.edge_attr))
        )
        h = h1 + message2 if MODEL_CONFIG.residual_connections else message2

        h_a = h[player_a_idx]
        h_b = h[player_b_idx]
        correction_ab = self.pair_score(h_a, h_b, match_context)

        if MODEL_CONFIG.antisymmetric_decoder:
            correction_ba = self.pair_score(h_b, h_a, match_context)
            correction = 0.5 * (correction_ab - correction_ba)
        else:
            correction = correction_ab

        if MODEL_CONFIG.direct_bscore_logit:
            bscore_a = data.raw_bscore_general[player_a_idx]
            bscore_b = data.raw_bscore_general[player_b_idx]
            logits = self.bscore_scale * (bscore_a - bscore_b) + correction
        elif MODEL_CONFIG.antisymmetric_decoder:
            # No intercept is allowed here: it would break
            # logit(B, A) = -logit(A, B).
            logits = correction
        else:
            # Exact legacy behaviour retained for current_gnn and edge_full.
            logits = self.bscore_scale + correction

        return logits, (h_a - h_b).detach()


In [ ]:
model = TennisGINE(
    node_in_dim=6,
    edge_in_dim=len(edge_feature_names(EDGE_CONFIG)),
    hidden_dim= 32,
    match_context_dim=4,
    dropout = 0.3
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

model

In [ ]:
print(bscore_snapshots.columns.tolist())
print(bscore_snapshots.head())

In [ ]:
def make_replay_batch(replay_buffer, device):
    graphs = [item[0] for item in replay_buffer]
    batch = Batch.from_data_list(graphs).to(device)

    offsets = batch.ptr[:-1]  # start-row of each graph inside the batched x

    player_a = torch.cat([
        item[1] + offsets[i] for i, item in enumerate(replay_buffer)
    ]).to(device)
    player_b = torch.cat([
        item[2] + offsets[i] for i, item in enumerate(replay_buffer)
    ]).to(device)
    match_context = torch.cat([item[3] for item in replay_buffer]).to(device)
    y = torch.cat([item[4] for item in replay_buffer]).to(device)

    return batch, player_a, player_b, match_context, y

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score


bscore_base = pd.read_csv(PROCESSED_DIR / "atp_matches_with_bscore_and_intransitivity.csv")

bscore_base["tourney_date"] = pd.to_datetime(bscore_base["tourney_date"])

categories = (
    ["G"]
    if TOURNAMENT_SCOPE == "slams"
    else ["G", "M"]
)

bscore_base = bscore_base[
    (bscore_base["tourney_date"] >= START_DATE)
    & (bscore_base["tourney_level"].isin(categories))
].copy()

bscore_base = bscore_base.dropna(subset=[
    "winner_score_pre",
    "loser_score_pre",
    "tourney_date"
]).copy()

bscore_base["year"] = bscore_base["tourney_date"].dt.year
bscore_base["phase"] = bscore_base["year"].apply(assign_phase)

# raw diff: winner bscore - loser bscore
bscore_base["winner_loser_bscore_diff"] = (
    bscore_base["winner_score_pre"] - bscore_base["loser_score_pre"]
)

rng = np.random.default_rng(SEED)

# True = player A is the winner
a_is_winner = rng.random(len(bscore_base)) < 0.5

bscore_base["x_bscore_diff"] = np.where(
    a_is_winner,
    bscore_base["winner_loser_bscore_diff"],
    -bscore_base["winner_loser_bscore_diff"]
)

bscore_base["y"] = np.where(a_is_winner, 1, 0)

bscore_base[[
    "tourney_date",
    "phase",
    "winner_name",
    "loser_name",
    "winner_score_pre",
    "loser_score_pre",
    "x_bscore_diff",
    "y",
    "intransitivity_level"
]].head()
train_b = bscore_base[bscore_base["phase"] == "train"].copy()
val_b = bscore_base[bscore_base["phase"] == "val"].copy()
test_b = bscore_base[bscore_base["phase"] == "test"].copy()

print(len(train_b), len(val_b), len(test_b))
mu = train_b["x_bscore_diff"].mean()
sigma = train_b["x_bscore_diff"].std()

if sigma == 0 or np.isnan(sigma):
    sigma = 1.0

for df in [train_b, val_b, test_b]:
    df["x_bscore_diff_scaled"] = (df["x_bscore_diff"] - mu) / sigma

In [ ]:
rng = np.random.default_rng(SEED)

ROLLING_START_DATE = "2011-01-01"

current_history = matches[
    (matches["tourney_date"] <= "2011-01-01")
].copy()

logs = []
pred_rows = []

## wie viele optimizer updates nach jedem Blo
# ck
steps_per_block = 2

## blocks für stabilisation des online trainings
replay_buffer = []
replay_buffer_size = 200

WARMUP_END = 2011
UPDATE_PHASES =  {"train"}



for block_idx, block in rolling_blocks.iterrows():
    tourney_id = block["tourney_id"]
    round_order = block["round_order"]
    t_block = block["tourney_date"]
    phase = block["phase"]
    year = block["year"]

    block_matches = future_matches[
        (future_matches["tourney_id"] == tourney_id) &
        (future_matches["round_order"] == round_order)
    ].copy()

    block_matches_all = block_matches.copy()

    # During train and validation, use all matches.
    # During test, only evaluate high-intransitivity matches.
    block_matches_target = block_matches.copy()


    snap_dict, defaults = get_bscore_snapshots(tourney_id, round_order)

    ## graph snapshot mit nur past matches
    data,player_to_idx = build_pyg_snapshot(
        current_history=current_history, 
        block_matches= block_matches_all, 
        t_block=t_block, 
        snap_dict=snap_dict, 
        defaults=defaults, 
        alpha_days=365)

    target_tensors = make_block_targets(block_matches=block_matches_target, player_to_idx=player_to_idx, snap_dict = snap_dict, rng=rng, defaults = defaults)

    if target_tensors is None:
        current_history = pd.concat([current_history, block_matches_all], ignore_index=True)
        continue

    player_a, player_b, match_context, y, bscore_diffs, intransitivity_levels = target_tensors

    data= data.to(device)
    player_b = player_b.to(device)
    player_a = player_a.to(device)
    match_context = match_context.to(device)
    y = y.to(device)

    ## evaluate bevor wir auf dem Block trainieren
    model.eval()
    with torch.no_grad():
        logits_eval, emb_diff_tensor = model(data, player_a, player_b, match_context)
        emb_diff = emb_diff_tensor.cpu().numpy()
        probs_eval = torch.sigmoid(logits_eval)
        loss_before = criterion(logits_eval, y).item()
        preds = (probs_eval >= 0.5).float()
        acc_before = (preds == y).float().mean().item()

    probs_np = probs_eval.cpu().numpy()
    preds_np = preds.cpu().numpy()
    y_np = y.cpu().numpy()

    for i in range(len(y_np)):
        pred_rows.append({
            "tourney_id": tourney_id,
            "round_order": round_order,
            "tourney_date": t_block,
            "phase": phase,
            "year": year,
            "prob_before": float(probs_np[i]),
            "pred_before": float(preds_np[i]),
            "y_true": float(y_np[i]),
            "block_idx": block_idx,
            "row_in_block": i,
            "emb_diff": emb_diff[i].tolist(),
            "bscore_diff": float(bscore_diffs[i].item()),
            "intransitivity_level": intransitivity_levels[i]
        })

    if phase in UPDATE_PHASES and year > WARMUP_END:

        ## kopie des blocks im replay buffer
        data_buf = data.clone()
        data_buf.x = data_buf.x.detach()
        data_buf.edge_index = data_buf.edge_index.detach()
        data_buf.edge_attr = data_buf.edge_attr.detach()

        replay_buffer.append((
            data_buf,
            player_a.detach(),
            player_b.detach(),
            match_context.detach(),
            y.detach()
        ))

        if len(replay_buffer) > replay_buffer_size:
            replay_buffer = replay_buffer[-replay_buffer_size:]

        model.train()
        for _ in range(steps_per_block):
            optimizer.zero_grad()
            batch, ra, rb, rctx, ry = make_replay_batch(replay_buffer, device)
            logits, _ = model(batch, ra, rb, rctx)
            loss = criterion(logits, ry)
            loss.backward()
            optimizer.step()

        ##evaluation nach dem update
        model.eval()

        with torch.no_grad():
            logits_after, _ = model(data, player_a, player_b, match_context)
            probs_after = torch.sigmoid(logits_after)

            loss_after = criterion(logits_after, y).item()
            preds_after = (probs_after >= 0.5).float()
            acc_after = (preds_after == y).float().mean().item()
    else:
        loss_after = np.nan
        acc_after = np.nan


    ## die after stats sagen uns eher ob die GNN gerade irgendwas macht oder nicht, sie sind nicht metriken die uns viel sagen
    logs.append({
        "block_idx": block_idx,
        "tourney_id": tourney_id,
        "phase": phase,
        "year": year,
        "round_order": round_order,
        "num_matches": len(block_matches),
        "loss_before": loss_before,
        "acc_before": acc_before,
        "loss_after": loss_after,
        "acc_after": acc_after,
        "num_nodes": data.num_nodes,
        "num_edges": data.num_edges
    })
    
    current_history = pd.concat([current_history, block_matches_all], ignore_index=True)

    if block_idx %20 == 0:
        print(
            f"({block_idx:03d} {tourney_id} {block['round_order']}) "
            f"loss_before={loss_before:.4f} acc_before={acc_before:.3f} "
            f"loss_after={loss_after:.4f} acc_after={acc_after:.3f}"
        )
    

In [ ]:
logs_df = pd.DataFrame(logs)
predictions_df = pd.DataFrame(pred_rows)

logs_df.head()

In [ ]:


def phase_metrics(df):
    y = df["y_true"].to_numpy()
    p = df["prob_before"].to_numpy()
    pred = df["pred_before"].to_numpy()

    eps = 1e-7
    p = np.clip(p, eps, 1- eps)
    
    accuracy = (pred==y).mean()
    logloss= -np.mean(y*np.log(p) + (1-y)*np.log(1-p))
    brier= np.mean((p-y)**2)
    # auc = roc_auc_score(y, p)

    return pd.Series({
        "n_matches": len(df),
        "accuracy": accuracy,
        "logloss" : logloss,
        "brier": brier,
        # "auc": auc
    })

metrics_by_phase = (
    predictions_df
    .groupby("phase")
    .apply(phase_metrics)
)

metrics_by_phase


In [ ]:
test_predictions = predictions_df[
    predictions_df["phase"] == "test"
].copy()

metrics_by_intransitivity = (
    test_predictions
    .groupby("intransitivity_level")
    .apply(phase_metrics)
    .reindex(["low", "medium", "high"])
)

metrics_by_intransitivity

In [ ]:
test_b_gnn_subset = predictions_df[predictions_df["phase"] == "test"].copy()

test_b_gnn_subset["x_bscore_diff"] = test_b_gnn_subset["bscore_diff"]
test_b_gnn_subset["y"] = test_b_gnn_subset["y_true"]

test_b_gnn_subset["x_bscore_diff_scaled"] = (
    test_b_gnn_subset["x_bscore_diff"] - mu
) / sigma

print("GNN test rows:", len(test_b_gnn_subset))


In [ ]:
def evaluate_bscore_rule(df):
    y_true = df["y"].to_numpy()
    y_pred = (df["x_bscore_diff"] >= 0).astype(int)

    return pd.Series({
        "n_matches": len(df),
        "accuracy": accuracy_score(y_true, y_pred)
    })

rule_results = pd.DataFrame({
    "train": evaluate_bscore_rule(train_b),
    "val": evaluate_bscore_rule(val_b),
    "test": evaluate_bscore_rule(test_b_gnn_subset)
}).T

rule_results

In [ ]:
X_train = train_b[["x_bscore_diff_scaled"]]
y_train = train_b["y"]

bscore_logit = LogisticRegression()
bscore_logit.fit(X_train, y_train)

In [ ]:
def evaluate_prob_model(model, df):
    X = df[["x_bscore_diff_scaled"]]
    y_true = df["y"].to_numpy()

    prob = model.predict_proba(X)[:, 1]
    pred = (prob >= 0.5).astype(int)

    return pd.Series({
        "n_matches": len(df),
        "accuracy": accuracy_score(y_true, pred),
        "log_loss": log_loss(y_true, prob),
        "brier": brier_score_loss(y_true, prob),
        "auc": roc_auc_score(y_true, prob)
    })

bscore_logit_results = pd.DataFrame({
    "train": evaluate_prob_model(bscore_logit, train_b),
    "val": evaluate_prob_model(bscore_logit, val_b),
    "test": evaluate_prob_model(bscore_logit, test_b_gnn_subset)
}).T

bscore_logit_results

In [ ]:
comparison = pd.DataFrame({
    "GNN_accuracy": metrics_by_phase["accuracy"],
    "GNN_log_loss": metrics_by_phase["logloss"],
    "GNN_brier": metrics_by_phase["brier"],

    "BscoreLogit_accuracy": bscore_logit_results["accuracy"],
    "BscoreLogit_log_loss": bscore_logit_results["log_loss"],
    "BscoreLogit_brier": bscore_logit_results["brier"],
    "BscoreLogit_auc": bscore_logit_results["auc"],
})

comparison

In [ ]:
test_predictions = predictions_df[
    predictions_df["phase"] == "test"
].copy()

test_predictions["x_bscore_diff_scaled"] = (
    test_predictions["bscore_diff"] - mu
) / sigma

test_predictions["bscore_prob"] = bscore_logit.predict_proba(
    test_predictions[["x_bscore_diff_scaled"]]
)[:, 1]

test_predictions["bscore_pred"] = (
    test_predictions["bscore_prob"] >= 0.5
).astype(int)

In [ ]:
def metrics_by_level(df):
    y = df["y_true"].to_numpy()

    gnn_prob = np.clip(
        df["prob_before"].to_numpy(),
        1e-7,
        1 - 1e-7
    )
    gnn_pred = df["pred_before"].to_numpy()

    bscore_prob = np.clip(
        df["bscore_prob"].to_numpy(),
        1e-7,
        1 - 1e-7
    )
    bscore_pred = df["bscore_pred"].to_numpy()

    return pd.Series({
        "n_matches": len(df),

        "gnn_accuracy": accuracy_score(y, gnn_pred),
        "bscore_accuracy": accuracy_score(y, bscore_pred),

        "gnn_log_loss": log_loss(y, gnn_prob, labels=[0, 1]),
        "bscore_log_loss": log_loss(
            y,
            bscore_prob,
            labels=[0, 1]
        ),

        "gnn_brier": brier_score_loss(y, gnn_prob),
        "bscore_brier": brier_score_loss(
            y,
            bscore_prob
        )
    })

comparison_by_intransitivity = (
    test_predictions
    .groupby("intransitivity_level")
    .apply(metrics_by_level)
    .reindex(["low", "medium", "high"])
)

comparison_by_intransitivity

In [ ]:
from math import erf, sqrt

try:
    from scipy.stats import t as student_t
except Exception:
    student_t = None


def diebold_mariano_test(loss_model_1, loss_model_2, h=1, alternative="two-sided"):
    loss_model_1 = np.asarray(loss_model_1, dtype=float)
    loss_model_2 = np.asarray(loss_model_2, dtype=float)

    if loss_model_1.shape != loss_model_2.shape:
        raise ValueError("Le due serie di loss devono avere la stessa lunghezza.")

    d = loss_model_1 - loss_model_2
    d = d[np.isfinite(d)]
    T = len(d)

    if T <= h:
        raise ValueError("Troppi pochi dati per il Diebold-Mariano test con questo orizzonte h.")

    d_bar = d.mean()
    centered = d - d_bar

    gamma0 = np.dot(centered, centered) / T
    long_run_var = gamma0

    for lag in range(1, h):
        gamma = np.dot(centered[lag:], centered[:-lag]) / T
        long_run_var += 2 * (1 - lag / h) * gamma

    if long_run_var <= 0:
        raise ValueError("La long-run variance non e' positiva; impossibile calcolare il test.")

    dm_stat = d_bar / np.sqrt(long_run_var / T)

    harvey_adj = np.sqrt((T + 1 - 2 * h + (h * (h - 1)) / T) / T)
    dm_stat *= harvey_adj

    if student_t is not None:
        if alternative == "two-sided":
            p_value = 2 * student_t.sf(abs(dm_stat), df=T - 1)
        elif alternative == "less":
            p_value = student_t.cdf(dm_stat, df=T - 1)
        elif alternative == "greater":
            p_value = student_t.sf(dm_stat, df=T - 1)
        else:
            raise ValueError("alternative deve essere 'two-sided', 'less' o 'greater'.")
    else:
        normal_cdf = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
        if alternative == "two-sided":
            p_value = 2 * (1 - normal_cdf(abs(dm_stat)))
        elif alternative == "less":
            p_value = normal_cdf(dm_stat)
        elif alternative == "greater":
            p_value = 1 - normal_cdf(dm_stat)
        else:
            raise ValueError("alternative deve essere 'two-sided', 'less' o 'greater'.")

    return pd.Series({
        "n_matches": T,
        "mean_loss_diff": d_bar,
        "dm_stat": dm_stat,
        "p_value": p_value
    })


test_dm = predictions_df[predictions_df["phase"] == "test"].copy()

X_test_bscore = pd.DataFrame({
    "x_bscore_diff_scaled": (test_dm["bscore_diff"] - mu) / sigma
})

eps = 1e-7
y_test_dm = test_dm["y_true"].to_numpy()
gnn_prob = np.clip(test_dm["prob_before"].to_numpy(), eps, 1 - eps)
bscore_prob = np.clip(bscore_logit.predict_proba(X_test_bscore)[:, 1], eps, 1 - eps)

gnn_logloss = -(y_test_dm * np.log(gnn_prob) + (1 - y_test_dm) * np.log(1 - gnn_prob))
bscore_logloss = -(y_test_dm * np.log(bscore_prob) + (1 - y_test_dm) * np.log(1 - bscore_prob))

gnn_brier = (gnn_prob - y_test_dm) ** 2
bscore_brier = (bscore_prob - y_test_dm) ** 2

dm_results = pd.DataFrame({
    "log_loss": diebold_mariano_test(gnn_logloss, bscore_logloss),
    "brier": diebold_mariano_test(gnn_brier, bscore_brier)
}).T

dm_results.index.name = "loss"
dm_results